# FakeGuard AI — 3-Class Fake Image Detection Training
**EfficientNet-B4 on Google Colab T4 GPU**

Classes: `REAL (0)` | `AI_GENERATED (1)` | `EDITED (2)`

Checkpoint saved at: `efficientnet_b4_fake.pth` → place in `backend/checkpoints/`


## 1 — Check GPU & Install Dependencies

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    raise RuntimeError('No GPU! Go to Runtime → Change runtime type → T4 GPU')

In [ ]:
!pip install -q kaggle scikit-image
print('Done.')

## 2 — Kaggle Setup

Upload `kaggle.json` from kaggle.com → Account → API → Create New Token

In [ ]:
from google.colab import files
import os

print('Upload kaggle.json:')
up = files.upload()
os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json', 'w') as f:
    f.write(list(up.values())[0].decode())
os.chmod('/root/.kaggle/kaggle.json', 0o600)
print('Kaggle API ready.')

## 3 — Download Datasets
- **Real + AI_Generated**: 140k Real and Fake Faces (StyleGAN) — ~2.5 GB
- **Edited**: CASIA v2 (copy-move + splicing) — ~1 GB

In [ ]:
!kaggle datasets download -d xhlulu/140k-real-and-fake-faces -p /content/dl_140k --unzip -q
print('140k dataset downloaded.')

In [ ]:
!kaggle datasets download -d sophatvathana/casia-dataset -p /content/dl_casia --unzip -q
print('CASIA dataset downloaded.')

## 4 — Organize Dataset Structure

In [ ]:
import shutil, random
from pathlib import Path
from PIL import Image
import numpy as np

EXTS = {'.jpg', '.jpeg', '.png', '.webp'}
DATA = Path('/content/data')
MAX = 8000  # images per class per split (reduce if disk full)

def copy_imgs(src, dst, limit=None):
    dst = Path(dst); dst.mkdir(parents=True, exist_ok=True)
    files = [f for f in Path(src).rglob('*') if f.suffix.lower() in EXTS]
    if limit: files = random.sample(files, min(limit, len(files)))
    for f in files: shutil.copy2(f, dst / f.name)
    print(f'  {Path(dst).relative_to(DATA)}: {len(files):,}')
    return len(files)

# ── 140k: real + ai_generated ────────────────────────────────────────────
base = Path('/content/dl_140k/real_vs_fake/real-or-fake')
for src_split, dst_split in [('train','train'),('test','val')]:
    for src_cls, dst_cls in [('real','real'),('fake','ai_generated')]:
        src = base / src_split / src_cls
        if src.exists(): copy_imgs(src, DATA/dst_split/dst_cls, MAX)
        else: print(f'  WARNING: {src} not found')

# ── CASIA v2: edited ─────────────────────────────────────────────────────
casia = Path('/content/dl_casia')
tampered = []
for name in ['Tp', 'tampered', 'CASIA2', 'fake']:
    for d in casia.rglob(name):
        if d.is_dir():
            tampered += [f for f in d.rglob('*') if f.suffix.lower() in EXTS]

if tampered:
    random.shuffle(tampered)
    sp = int(len(tampered) * 0.8)
    for split, files in [('train', tampered[:sp]), ('val', tampered[sp:])]:
        dst = DATA / split / 'edited'; dst.mkdir(parents=True, exist_ok=True)
        chosen = files[:MAX]
        for f in chosen: shutil.copy2(f, dst / f.name)
        print(f'  {split}/edited: {len(chosen):,}')
else:
    print('CASIA Tp not found — generating synthetic edited images as fallback...')
    real_files = list((DATA / 'train' / 'real').glob('*.jpg'))[:3000]
    random.shuffle(real_files)
    sp = int(len(real_files) * 0.8)
    for split, portion in [('train', real_files[:sp]), ('val', real_files[sp:])]:
        dst = DATA / split / 'edited'; dst.mkdir(parents=True, exist_ok=True)
        for i in range(0, len(portion)-1, 2):
            a = np.array(Image.open(portion[i]).convert('RGB').resize((256,256)))
            b = np.array(Image.open(portion[i+1]).convert('RGB').resize((256,256)))
            spliced = a.copy(); spliced[:, 128:] = b[:, 128:]
            Image.fromarray(spliced).save(dst / f'splice_{i:05d}.jpg', quality=85)
        print(f'  {split}/edited: {len(portion)//2} synthetic')

print('\nDataset organized.')

In [ ]:
from pathlib import Path
DATA = Path('/content/data')
EXTS = {'.jpg', '.jpeg', '.png', '.webp'}

total = 0
print(f'  {"Split/Class":>20}  {"Count":>8}')
print('-' * 35)
for split in ['train', 'val']:
    for cls in ['real', 'ai_generated', 'edited']:
        d = DATA / split / cls
        n = len([f for f in d.rglob('*') if f.suffix.lower() in EXTS]) if d.exists() else 0
        s = '✓' if n >= 500 else ('⚠' if n > 0 else '✗')
        print(f'  {s}  {split}/{cls:14}  {n:8,}')
        total += n
print(f'\n  Total: {total:,} images')

## 5 — Model + Dataset + Training
**EfficientNet-B4** — same architecture as `backend/models/cnn_model.py`  
Checkpoint format matches what `blind_detector.py` expects.

In [ ]:
import torch, io, random
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models

CLASSES = ['REAL', 'AI_GENERATED', 'EDITED']
NUM_CLASSES = 3

class FakeImageDetector(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        w = models.EfficientNet_B4_Weights.IMAGENET1K_V1 if pretrained else None
        base = models.efficientnet_b4(weights=w)
        in_f = base.classifier[1].in_features
        base.classifier = nn.Sequential(
            nn.Dropout(0.4), nn.Linear(in_f, 512), nn.GELU(),
            nn.Dropout(0.2), nn.Linear(512, NUM_CLASSES),
        )
        self.backbone = base

    def forward(self, x): return self.backbone(x)

    def predict_proba(self, x):
        with torch.no_grad(): return F.softmax(self.forward(x), dim=1)

    def freeze_backbone(self):
        for n, p in self.backbone.named_parameters():
            if 'classifier' not in n: p.requires_grad = False

    def unfreeze_all(self):
        for p in self.backbone.parameters(): p.requires_grad = True

print('Model defined:', CLASSES)

In [ ]:
from pathlib import Path
from PIL import Image
import numpy as np
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision.transforms as T

IMG_SIZE = 380
EXTS = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}
LABEL_MAP = {'real': 0, 'ai_generated': 1, 'edited': 2}

class JPEGAug:
    def __call__(self, img):
        buf = io.BytesIO()
        img.save(buf, 'JPEG', quality=random.randint(55, 95))
        buf.seek(0)
        return Image.open(buf).convert('RGB')

class GaussNoise:
    def __call__(self, t):
        return (t + torch.randn_like(t) * random.uniform(0, 0.015)).clamp(0, 1)

NRM = T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
TRAIN_TF = T.Compose([
    T.Resize((IMG_SIZE+40, IMG_SIZE+40)), T.RandomCrop(IMG_SIZE),
    T.RandomHorizontalFlip(0.5), T.RandomRotation(12),
    T.ColorJitter(0.3,0.3,0.2,0.05), T.RandomGrayscale(0.05),
    JPEGAug(), T.ToTensor(), GaussNoise(), NRM,
])
VAL_TF = T.Compose([T.Resize((IMG_SIZE,IMG_SIZE)), T.ToTensor(), NRM])

class FakeDS(Dataset):
    def __init__(self, root, split='train'):
        base = Path(root) / split
        self.paths = []
        for cls, lbl in LABEL_MAP.items():
            d = base / cls
            if d.exists():
                imgs = [p for p in d.rglob('*') if p.suffix.lower() in EXTS]
                self.paths += [(p, lbl) for p in imgs]
                print(f'  [{split}] {cls}: {len(imgs):,}')
            else: print(f'  [{split}] {cls}: NOT FOUND')
        self.tf = TRAIN_TF if split == 'train' else VAL_TF

    def __len__(self): return len(self.paths)

    def __getitem__(self, i):
        p, lbl = self.paths[i]
        try: img = Image.open(p).convert('RGB')
        except: img = Image.new('RGB', (IMG_SIZE,IMG_SIZE), 128)
        return self.tf(img), torch.tensor(lbl, dtype=torch.long)

    def sampler(self):
        lbls = [l for _,l in self.paths]
        cnt = [lbls.count(c) for c in range(NUM_CLASSES)]
        w = [1.0/max(cnt[l],1) for l in lbls]
        return WeightedRandomSampler(w, len(w), replacement=True)

print('Dataset class defined.')

In [ ]:
import numpy as np
from torch.cuda.amp import GradScaler, autocast

def metrics(logits, labels):
    preds = logits.argmax(1)
    acc = (preds == labels).float().mean().item()
    f1s = []
    for c in range(NUM_CLASSES):
        tp = ((preds==c)&(labels==c)).sum().item()
        fp = ((preds==c)&(labels!=c)).sum().item()
        fn = ((preds!=c)&(labels==c)).sum().item()
        p = tp/(tp+fp+1e-8); r = tp/(tp+fn+1e-8)
        f1s.append(2*p*r/(p+r+1e-8))
    return {'acc': acc, 'macro_f1': float(np.mean(f1s))}

def train_one(model, loader, opt, crit, scaler, device):
    model.train()
    loss_sum, logs, labs = 0, [], []
    for step, (x, y) in enumerate(loader):
        x, y = x.to(device), y.to(device)
        opt.zero_grad()
        with autocast():
            out = model(x); loss = crit(out, y)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt); scaler.update()
        loss_sum += loss.item()
        logs.append(out.detach().cpu()); labs.append(y.cpu())
        if (step+1) % 100 == 0: print(f'   step {step+1}/{len(loader)} loss={loss.item():.4f}')
    m = metrics(torch.cat(logs), torch.cat(labs))
    m['loss'] = loss_sum / len(loader)
    return m

@torch.no_grad()
def eval_one(model, loader, crit, device):
    model.eval()
    loss_sum, logs, labs = 0, [], []
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        with autocast(): out = model(x); loss = crit(out, y)
        loss_sum += loss.item()
        logs.append(out.cpu()); labs.append(y.cpu())
    m = metrics(torch.cat(logs), torch.cat(labs))
    m['loss'] = loss_sum / len(loader)
    return m

print('Training functions defined.')

In [ ]:
# ── Configuration ─────────────────────────────────────────
DATA_DIR      = '/content/data'
CKPT_PATH     = '/content/efficientnet_b4_fake.pth'
EPOCHS        = 30
BATCH         = 32
LR_HEAD       = 1e-3
LR_FINE       = 2e-5
FREEZE_EPOCHS = 5
WORKERS       = 2
# ─────────────────────────────────────────────────────────

device = torch.device('cuda')

train_ds = FakeDS(DATA_DIR, 'train')
val_ds   = FakeDS(DATA_DIR, 'val')
print(f'\nTrain: {len(train_ds):,}  Val: {len(val_ds):,}')

train_loader = DataLoader(train_ds, BATCH, sampler=train_ds.sampler(),
                          num_workers=WORKERS, pin_memory=True)
val_loader   = DataLoader(val_ds, BATCH, shuffle=False,
                          num_workers=WORKERS, pin_memory=True)

model     = FakeImageDetector(pretrained=True).to(device)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
scaler    = GradScaler()

print('Setup complete.')

In [ ]:
import time

best_f1 = 0.0
scheduler = None
optimizer = None

for epoch in range(1, EPOCHS+1):
    print(f'\n{"="*65}')
    print(f'Epoch {epoch}/{EPOCHS}')

    if epoch <= FREEZE_EPOCHS:
        if epoch == 1:
            model.freeze_backbone()
            print('Phase 1: head only (backbone frozen)')
        optimizer = torch.optim.AdamW(
            filter(lambda p: p.requires_grad, model.parameters()),
            lr=LR_HEAD, weight_decay=1e-4)
    elif epoch == FREEZE_EPOCHS + 1:
        model.unfreeze_all()
        print('Phase 2: full fine-tuning')
        optimizer = torch.optim.AdamW(model.parameters(), lr=LR_FINE, weight_decay=1e-4)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=EPOCHS-FREEZE_EPOCHS, eta_min=1e-7)

    t0 = time.time()
    tr = train_one(model, train_loader, optimizer, criterion, scaler, device)
    vl = eval_one(model, val_loader, criterion, device)
    if scheduler: scheduler.step()

    dt = time.time() - t0
    print(f'Train | loss={tr["loss"]:.4f} acc={tr["acc"]:.4f} macro_f1={tr["macro_f1"]:.4f}')
    print(f'Val   | loss={vl["loss"]:.4f} acc={vl["acc"]:.4f} macro_f1={vl["macro_f1"]:.4f}  [{dt:.0f}s]')

    if vl['macro_f1'] > best_f1:
        best_f1 = vl['macro_f1']
        torch.save({'epoch': epoch, 'model': model.state_dict(),
                    'val_macro_f1': best_f1, 'val_acc': vl['acc'],
                    'classes': CLASSES}, CKPT_PATH)
        print(f'  ✓ Checkpoint saved (macro_f1={best_f1:.4f})')

print(f'\nDone! Best val macro-F1: {best_f1:.4f}')

## 6 — Evaluate (Per-Class Breakdown)

In [ ]:
import numpy as np

state = torch.load(CKPT_PATH, map_location=device)
model.load_state_dict(state['model'])
model.eval()

all_p, all_l = [], []
with torch.no_grad():
    for x, y in val_loader:
        with autocast(): out = model(x.to(device))
        all_p += out.argmax(1).cpu().tolist()
        all_l += y.tolist()

all_p = np.array(all_p); all_l = np.array(all_l)
acc = (all_p == all_l).mean()

conf = np.zeros((NUM_CLASSES, NUM_CLASSES), int)
for t, p in zip(all_l, all_p): conf[t][p] += 1

print('='*60, '\nEVALUATION', '\n'+'='*60)
print(f'Accuracy: {acc:.4f} ({acc*100:.1f}%)\n')
print(f'{"Class":>15}  {"Prec":>6}  {"Recall":>6}  {"F1":>6}  {"N":>6}')
print('-'*50)
for c in range(NUM_CLASSES):
    tp=conf[c,c]; fp=conf[:,c].sum()-tp; fn=conf[c,:].sum()-tp
    p=tp/(tp+fp+1e-8); r=tp/(tp+fn+1e-8); f1=2*p*r/(p+r+1e-8)
    print(f'{CLASSES[c]:>15}  {p:>6.4f}  {r:>6.4f}  {f1:>6.4f}  {conf[c,:].sum():>6}')
print('\nConfusion Matrix (True \ Pred):')
print(f'{"":>15}'+' '.join(f'{c:>15}' for c in CLASSES))
for i,c in enumerate(CLASSES):
    print(f'{c:>15}'+' '.join(f'{conf[i,j]:>15}' for j in range(NUM_CLASSES)))

## 7 — Download Checkpoint

Place `efficientnet_b4_fake.pth` in `backend/checkpoints/` then restart the backend.
It auto-loads — no config changes needed.

In [ ]:
from google.colab import files
import os

sz = os.path.getsize(CKPT_PATH) / 1e6
print(f'Checkpoint: {sz:.1f} MB')
print(f'val_macro_f1 = {state["val_macro_f1"]:.4f}')
print(f'val_acc      = {state["val_acc"]:.4f}')
print('\nPlace downloaded file at: backend/checkpoints/efficientnet_b4_fake.pth')
files.download(CKPT_PATH)

## Notes

### Expected Performance (30 epochs, T4)
| Class | Expected F1 |
|-------|-------------|
| REAL | 0.88–0.95 |
| AI_GENERATED | 0.90–0.96 |
| EDITED | 0.75–0.88 |
| **Macro** | **0.85–0.93** |

### If edited F1 is low (<0.70)
Add more edited images: Coverage, Columbia, NIST16 datasets.

### If Colab disconnects
The checkpoint is saved to `/content/` — download it immediately after training.